In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from ksb.simulation.ksb_simulation import KSBSimulation
from ksb.simulation.result import SimulationResult
from ksb.planning.solvers.scurve import SCurveSolver
from ksb.planning.solvers.quintic import QuinticSolver
from ksb.motion.trajectories import (
    CompositeTrajectory,
    ConstantJerkTrajectory,
    LinearTrajectory,
    PolynomialTrajectory,
    P, V, A,
)

import yaml
from pathlib import Path
with open(Path('..') / 'configs' / 'default.yaml') as f:
    cfg = yaml.safe_load(f)

In [2]:
# ── Run simulation and display ────────────────────────────────────────────────
sim = KSBSimulation(cfg=cfg, solver=SCurveSolver())
result = sim.run(seed=42)

# Extract timeline: list of (t_start, jerk, duration) tuples
timeline = sim._u_control._timeline

# Build piecewise constant jerk plot
t_plot = []
j_plot = []

for t_start, jerk, duration in timeline:
    t_end = t_start + duration
    # Add segment: horizontal line from start to end at jerk value
    t_plot.extend([t_start, t_end])
    j_plot.extend([jerk, jerk])

# Create figure with subplots for jerk, acceleration, velocity
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# Plot jerk timeline
axes[0].step(t_plot, j_plot, where='post', linewidth=2, color='tab:blue', label='Jerk')
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[0].set_ylabel('Jerk (m/s³)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Integrate timeline to get acceleration profile
t_accel = []
a_vals = []
a_current = 0.0
for t_start, jerk, duration in timeline:
    t_accel.append(t_start)
    a_vals.append(a_current)
    a_current += jerk * duration
    t_accel.append(t_start + duration)
    a_vals.append(a_current)

axes[1].step(t_accel, a_vals, where='post', linewidth=2, color='tab:orange', label='Acceleration')
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.3)
axes[1].set_ylabel('Acceleration (m/s²)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Integrate acceleration to get velocity profile
t_vel = []
v_vals = []
v_current = sim.vu
for i in range(len(t_accel) - 1):
    t_vel.append(t_accel[i])
    v_vals.append(v_current)
    # Integrate: v += a * dt
    if i < len(a_vals) - 1:
        dt = t_accel[i + 1] - t_accel[i]
        a_mid = a_vals[i]
        v_current += a_mid * dt

axes[2].step(t_vel, v_vals, where='post', linewidth=2, color='tab:green', label='Velocity')
axes[2].axhline(y=sim.vu, color='k', linestyle='--', linewidth=0.5, alpha=0.5, label=f'v_u = {sim.vu:.3f} m/s')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Velocity (m/s)')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Total timeline duration: {timeline[-1][0] + timeline[-1][2]:.2f} s")
print(f"Number of segments: {len(timeline)}")
print(f"Final velocity: {v_current:.3f} m/s")
print(f"Nominal upstream velocity: {sim.vu:.3f} m/s")

SlotAssignmentError: No feasible slot for input at t_control_start=22.4356